# Raven AI + NVIDIA Nemotron Integration

This notebook demonstrates how to integrate NVIDIA Nemotron models (via `langchain-nvidia-ai-endpoints`) with the Raven AI ecosystem for sovereign, local-first agentic AI workflows in biology and healthcare.

## Prerequisites

```bash
pip install langchain-nvidia-ai-endpoints raven-ai
# or from source:
pip install -e /path/to/raven-ai
```

## Setup

You'll need an NVIDIA API key from the [NVIDIA API Catalog](https://build.nvidia.com/).

In [ ]:
import os
import json

# Set your NVIDIA API key
# os.environ["NVIDIA_API_KEY"] = "nvapi-..."  # Replace with your key

if "NVIDIA_API_KEY" not in os.environ:
    print("⚠️  NVIDIA_API_KEY not set. Please set it to run the examples.")
else:
    print(f"✅ NVIDIA_API_KEY configured (length: {len(os.environ['NVIDIA_API_KEY'])})")

## 1. Basic Nemotron 3 Super Chat

Nemotron 3 Super (53B MoE, 12B active) is NVIDIA's cost-effective model for agentic workflows with tool calling.

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

# Nemotron 3 Super - cost-effective agentic model
llm = ChatNVIDIA(
    model="nvidia/nemotron-3-super-53b-a8b",
    max_tokens=4096,
    temperature=0.1,
)

response = llm.invoke(
    "Plan a 3-step agentic workflow for competitive research in oncology drug discovery."
)
print(response.content)

## 2. Nemotron 3 Ultra (Flagship) with Thinking Mode

Nemotron 3 Ultra (120B MoE, 12B active) is NVIDIA's flagship for agentic AI with 1M context.

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

# Nemotron 3 Ultra - flagship agentic model
llm_ultra = ChatNVIDIA(
    model="nvidia/nemotron-3-ultra-120b-a12b",
    max_tokens=8192,
    temperature=0.1,
)

# Enable thinking mode for complex reasoning
response = llm_ultra.invoke(
    "Think step by step: Design a clinical trial protocol for a novel CAR-T therapy targeting CD19+ B-cell malignancies. Include endpoints, stratification, and statistical considerations."
)
print(response.content[:2000] + "..." if len(response.content) > 2000 else response.content)

## 3. Raven AI Token Economy Integration

Raven AI's Token Economy plans model usage with explicit cost tracking. Here's how to integrate NVIDIA API costs.

In [ ]:
from runtime.token_economy import (
    TokenEconomyRequest,
    plan_token_economy,
    estimate_nvidia_cost,
)

# Plan a task with NVIDIA Nemotron 3 Super as the draft model
request = TokenEconomyRequest(
    task="agentic literature review for oncology biomarkers",
    complexity="standard",
    risk="medium",
    privacy="public",
    estimated_context_tokens=8000,
    estimated_output_tokens=2048,
    cache_hit_ratio=0.3,
    draft_confidence=0.7,
    evidence_coverage=0.6,
    tool_available=True,
    local_model_available=False,
    requires_exact_citations=True,
)

plan = plan_token_economy(request)
print(f"Draft Lane: {plan.draft_lane}")
print(f"Thinking Level: {plan.thinking_level}")
print(f"Context Budget: {plan.context_budget} tokens")
print(f"Draft Budget: {plan.draft_token_budget} tokens")
print(f"Verification Budget: {plan.verification_token_budget} tokens")
print(f"Max Output: {plan.max_output_tokens} tokens")
print(f"Estimated Cost: ${plan.estimated_cost_usd:.6f}")
print(f"Model: {plan.model_id}")
print(f"Actions: {plan.actions}")
print(f"Reason: {plan.reason}")

## 4. Cost Estimation for NVIDIA API

Direct cost estimation for budgeting.

In [ ]:
# Estimate costs for different NVIDIA models
models = [
    "nvidia/nemotron-3-ultra-120b-a12b",
    "nvidia/nemotron-3-super-53b-a8b",
    "nvidia/nv-embed-qa",
    "nvidia/nv-rerank-qa",
]

input_tokens = 10000
output_tokens = 2000

print(f"{'Model':<45} {'Input':>10} {'Output':>10} {'Total':>10} {'vs GPT-5.5':>12} {'vs Opus 4.7':>12}")
print("-" * 105)

for model in models:
    cost = estimate_nvidia_cost(model, input_tokens, output_tokens)
    print(f"{cost['model']:<45} ${cost['input_cost_usd']:>9.4f} ${cost['output_cost_usd']:>9.4f} ${cost['total_cost_usd']:>9.4f} ${cost['savings_vs_gpt55_usd']:>+11.4f} ${cost['savings_vs_opus47_usd']:>+11.4f}")

## 5. RAG with NV-Embed-QA and NV-Rerank-QA

Use NVIDIA's state-of-the-art embeddings and reranker for precision RAG.

In [ ]:
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings, NVIDIARerank
from langchain_core.documents import Document

# Embeddings for RAG retrieval
embedder = NVIDIAEmbeddings(model="nvidia/nv-embed-qa")

# Sample documents
docs = [
    Document(page_content="CAR-T cell therapy targets CD19 on B-cells for B-cell malignancies.", metadata={"source": "review-1"}),
    Document(page_content="CD22-directed CAR-T shows promise for CD19-negative relapse.", metadata={"source": "review-2"}),
    Document(page_content="CRISPR-engineered T-cells with PD-1 knockout enhance persistence.", metadata={"source": "review-3"}),
]

# Embed documents
embeddings = embedder.embed_documents([d.page_content for d in docs])
print(f"Embedded {len(docs)} documents, dimension: {len(embeddings[0])}")

# Reranker for precision
reranker = NVIDIARerank(model="nvidia/nv-rerank-qa")

query = "What are emerging targets for CAR-T therapy resistance?"
reranked = reranker.compress_documents(query=query, documents=docs)

print(f"\nReranked results for: {query}")
for i, doc in enumerate(reranked):
    print(f"  {i+1}. [{doc.metadata['source']}] {doc.page_content[:80]}...")

## 6. Self-Hosted NIM Integration (Local-First)

For sovereign deployments with PHI/private data, use self-hosted NIM with TensorRT-LLM optimization.

In [ ]:
# Configuration for self-hosted NIM (requires NVIDIA AI Enterprise)
import os

# Local NIM endpoint (update for your deployment)
NIM_BASE_URL = os.getenv("NIM_BASE_URL", "http://localhost:8000/v1")
NIM_MODEL = os.getenv("NIM_MODEL", "nvidia/nemotron-3-ultra-120b-a12b")

# Example: Using local NIM with ChatNVIDIA
# llm_local = ChatNVIDIA(
#     model=NIM_MODEL,
#     base_url=NIM_BASE_URL,
#     api_key="dummy",  # Not needed for local NIM
#     max_tokens=4096,
# )

print(f"Local NIM endpoint: {NIM_BASE_URL}")
print(f"Model: {NIM_MODEL}")
print("\nTo deploy local NIM:")
print("1. Obtain NVIDIA AI Enterprise license")
print("2. Pull NIM container: docker pull nvcr.io/nim/nvidia/nemotron-3-ultra:latest")
print("3. Run: docker run -d --gpus all -p 8000:8000 nvcr.io/nim/nvidia/nemotron-3-ultra:latest")
print("4. Set NIM_BASE_URL=http://localhost:8000/v1 and NIM_MODEL=nvidia/nemotron-3-ultra-120b-a12b")

## 7. Tool-Calling Agent with Nemotron

Nemotron models excel at tool-calling. Here's a PubMed search agent.

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.tools import tool
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate

# Define tools
@tool
def search_pubmed(query: str, limit: int = 5) -> str:
    """Search PubMed for biomedical literature."""
    # Placeholder - integrate with NCBI E-utilities in production
    return f"PubMed results for: {query} (limit: {limit}) - [Integrate with NCBI E-utilities API]"

@tool
def fetch_drug_info(drug_name: str) -> str:
    """Fetch drug information from DrugBank."""
    # Placeholder - integrate with DrugBank API in production
    return f"Drug info for: {drug_name} - [Integrate with DrugBank API]"

# Create agent with Nemotron 3 Super
llm = ChatNVIDIA(
    model="nvidia/nemotron-3-super-53b-a8b",
    max_tokens=4096,
    temperature=0.1,
)

tools = [search_pubmed, fetch_drug_info]

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are Raven AI — a biomedical research agent. Use tools to find evidence. Cite sources."),
    ("user", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Example query
# result = agent_executor.invoke({"input": "Find recent clinical trials for CD19 CAR-T in relapsed/refractory DLBCL"})
# print(result["output"])

print("Agent configured. Uncomment the invoke call to run (requires NVIDIA_API_KEY).")

## 8. Raven AI Evidence Graph Integration

Track provenance of NVIDIA model outputs with Raven's Evidence Graph.

In [ ]:
from runtime.evidence_graph import EvidenceGraph

# Create evidence graph
graph = EvidenceGraph()

# Add source
source = graph.add_source(
    title="Nemotron 3 Ultra Agentic Workflow Output",
    kind="model_output",
    metadata={"model": "nvidia/nemotron-3-ultra-120b-a12b", "task": "clinical_trial_design"}
)

# Add claim with confidence
claim = graph.add_claim(
    "CAR-T therapy with CD19 targeting shows 80% ORR in relapsed/refractory DLBCL.",
    [source.id],
    confidence=0.85,
    risk="medium",
    evidence_label="supported"
)

# Trace answer
trace = graph.trace_answer(
    "What is the efficacy of CD19 CAR-T in r/r DLBCL?",
    [claim.id]
)

print(f"Trace ID: {trace['trace_id']}")
print(f"Claims: {len(trace['claims'])}")
print(f"Sources: {len(trace['sources'])}")
print(f"Overall Confidence: {trace['confidence']:.2f}")

## 9. Scientific Agent Gates for NVIDIA Outputs

Before publishing NVIDIA model outputs, run through Raven's Scientific Agent Gates.

In [ ]:
from runtime.scientific_agent_gates import (
    ScientificRunManifest,
    evaluate_scientific_run,
)

# Create a manifest for NVIDIA model output
manifest = ScientificRunManifest(
    run_id="nvidia-nemotron-3-ultra-001",
    task_id="clinical-trial-design-001",
    question="Design a phase II clinical trial for CD19 CAR-T in r/r DLBCL",
    hypothesis="Nemotron 3 Ultra can generate a scientifically valid clinical trial protocol",
    workflow_stage="protocol_design",
    evidence_decision="supported",
    code_artifacts=["notebooks/clinical_trial_design.py"],
    output_artifacts=["protocols/cd19_cart_phase2.pdf"],
    metrics={"evidence_coverage": 0.88, "token_savings": 0.62},
    replay_command="python notebooks/clinical_trial_design.py",
    environment_fingerprint="python-3.11,nemotron-3-ultra,langchain-0.3.0",
    phi_routing_decision="local_only",
    human_review_status="pending",
)

report = evaluate_scientific_run(manifest)
print(f"Status: {report.status}")
print(f"Gate Decisions: {report.gate_decisions}")
print(f"Recommendations: {report.recommendations}")

## 10. Complete Workflow: Biology Research Agent

Putting it all together: a biology research agent with Nemotron 3 Ultra, Evidence Graph, Token Economy, and Scientific Gates.

In [ ]:
# Complete biology research workflow

async def biology_research_workflow(query: str, tenant_id: str = "research-lab-001"):
    """Run a complete biology research workflow with Raven AI + NVIDIA Nemotron."""
    
    # 1. Token Economy Planning
    from runtime.token_economy import TokenEconomyRequest, plan_token_economy
    
    econ_request = TokenEconomyRequest(
        task=query,
        complexity="research",
        risk="medium",
        privacy="public",
        estimated_context_tokens=16000,
        estimated_output_tokens=4096,
        cache_hit_ratio=0.2,
        draft_confidence=0.6,
        evidence_coverage=0.5,
        requires_exact_citations=True,
    )
    
    plan = plan_token_economy(econ_request)
    print(f"📋 Token Economy Plan: {plan.draft_lane} → {plan.thinking_level}")
    print(f"   Estimated cost: ${plan.estimated_cost_usd:.4f}")
    
    # 2. Run with NVIDIA Nemotron 3 Ultra
    from langchain_nvidia_ai_endpoints import ChatNVIDIA
    
    llm = ChatNVIDIA(
        model="nvidia/nemotron-3-ultra-120b-a12b",
        max_tokens=plan.max_output_tokens,
        temperature=0.1,
    )
    
    # Build prompt with evidence requirements
    prompt = f"""You are Raven AI — a biomedical research agent.
    Research Question: {query}
    
    Requirements:
    - Provide step-by-step reasoning
    - Cite specific sources (PubMed, clinicaltrials.gov, etc.)
    - Include confidence levels for each claim
    - Flag any speculative statements
    - Structured output with: hypothesis, methods, expected outcomes, risks
    """
    
    response = await llm.ainvoke(prompt.format(query=query))
    
    # 3. Evidence Graph
    from runtime.evidence_graph import EvidenceGraph
    
    graph = EvidenceGraph()
    source = graph.add_source(
        title=f"Nemotron 3 Ultra Output: {query[:50]}...",
        kind="model_output",
        metadata={"model": "nvidia/nemotron-3-ultra-120b-a12b", "tenant": tenant_id}
    )
    
    # Extract claims from response (simplified)
    claim = graph.add_claim(
        response.content[:500] + "...",
        [source.id],
        confidence=0.8,
        risk="medium"
    )
    
    trace = graph.trace_answer(query, [claim.id])
    
    # 4. Scientific Agent Gates
    from runtime.scientific_agent_gates import ScientificRunManifest, evaluate_scientific_run
    
    manifest = ScientificRunManifest(
        run_id=f"bio-research-{tenant_id}-{hash(query) % 10000}",
        task_id=f"task-{hash(query) % 10000}",
        question=query,
        hypothesis="Nemotron 3 Ultra can produce evidence-linked biomedical research output",
        workflow_stage="literature_review",
        evidence_decision="supported",
        code_artifacts=[],
        output_artifacts=[f"outputs/{manifest.run_id}.md"],
        metrics={"evidence_coverage": 0.75, "token_savings": 0.55},
        replay_command=f"python workflows/biology_research.py '{query}'",
        environment_fingerprint="python-3.11,nemotron-3-ultra,langchain-nvidia-ai-endpoints",
        phi_routing_decision="public",
        human_review_status="pending",
    )
    
    gate_report = evaluate_scientific_run(manifest)
    
    return {
        "response": response.content,
        "token_economy": {
            "draft_lane": plan.draft_lane,
            "thinking_level": plan.thinking_level,
            "estimated_cost_usd": plan.estimated_cost_usd,
            "model": plan.model_id,
        },
        "evidence_trace": trace,
        "gate_report": {
            "status": gate_report.status,
            "gate_decisions": gate_report.gate_decisions,
        },
    }

# Example usage (commented out - requires NVIDIA_API_KEY)
# import asyncio
# result = asyncio.run(biology_research_workflow(
#     "What are the mechanisms of resistance to CD19 CAR-T therapy in DLBCL?"
# ))
# print(json.dumps(result, indent=2))

print("Workflow defined. Uncomment and run with NVIDIA_API_KEY to execute.")

## Summary

This integration provides:

1. **Nemotron 3 Ultra/Super** — NVIDIA's flagship agentic models via `ChatNVIDIA`
2. **Token Economy** — Explicit cost tracking with NVIDIA API pricing
3. **Evidence Graph** — Provenance tracking for all model outputs
4. **Scientific Gates** — Publish/Review/Block decisions for scientific rigor
5. **Self-Hosted NIM** — Local-first deployment for PHI/sovereign data
5. **RAG Embeddings/Reranking** — NV-Embed-QA + NV-Rerank-QA for precision retrieval
6. **Tool-Calling Agents** — Native function calling with PubMed, DrugBank, etc.

All components work together to make Raven AI a sovereign, verifiable, cost-transparent platform for biomedical AI — powered by NVIDIA's best-in-class agentic models.